# 06 | Random Forest and XGBoost

**Goal:** compare nonlinear tree ensembles against Ridge under the family-held-out protocol. Random Forest averages bootstrapped trees; XGBoost adds shallow trees sequentially under a regularised loss. Hyperparameters are fixed in `src/train_models.py`; no held-out fold was used for tuning.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'; RESULTS = ROOT / 'results'
import sys; sys.path.insert(0, str(ROOT / 'src'))
df = pd.read_csv(DATA / 'layout_level_dataset.csv')
target = 'target_U_canopy_mean_m_s'
allm = pd.read_csv(RESULTS/'metrics_all_protocols.csv')
cols = ['protocol','model','mae_m_s','rmse_m_s','r2','mae_ci_low','mae_ci_high']
print(allm[(allm.feature_set=='engineered') & allm.model.isin(['Random Forest','XGBoost'])][cols].round(3).to_string(index=False))
print('\nRandom Forest: 500 trees; min_samples_leaf=2; max_features=0.9; seed=42')
print('XGBoost: 250 trees; depth=2; lr=0.03; subsample=0.85; colsample_bytree=0.9; L2=2.0; seed=42')

    protocol         model  mae_m_s  rmse_m_s    r2  mae_ci_low  mae_ci_high
family_5fold Random Forest    0.255     0.378 0.857       0.177        0.339
family_5fold       XGBoost    0.224     0.293 0.914       0.176        0.278
  family_loo Random Forest    0.248     0.369 0.864       0.166        0.334
  family_loo       XGBoost    0.228     0.297 0.912       0.181        0.278
layout_5fold Random Forest    0.070     0.112 0.987       0.057        0.088
layout_5fold       XGBoost    0.144     0.194 0.963       0.113        0.175

Random Forest: 500 trees; min_samples_leaf=2; max_features=0.9; seed=42
XGBoost: 250 trees; depth=2; lr=0.03; subsample=0.85; colsample_bytree=0.9; L2=2.0; seed=42


## The ranking changes with the split

Under the layout split Random Forest looked much better than XGBoost (MAE 0.070 vs 0.144). That advantage came from sibling layouts in training. With whole families held out the gap closes and reverses slightly. A paired family-level bootstrap of the difference:

In [2]:
print(json.dumps(json.loads((RESULTS/'rf_vs_xgb_paired.json').read_text()), indent=1))
print('\nPositive = XGBoost has lower MAE. Intervals containing 0 mean the two models are not distinguishable on this data.')

[
 {
  "protocol": "family_5fold",
  "mae_diff_rf_minus_xgb": 0.03162690743313868,
  "ci95_low": -0.012917394123264606,
  "ci95_high": 0.07786279265016255
 },
 {
  "protocol": "family_loo",
  "mae_diff_rf_minus_xgb": 0.020035827463794023,
  "ci95_low": -0.021646064050233095,
  "ci95_high": 0.06477468046145943
 },
 {
  "protocol": "layout_5fold",
  "mae_diff_rf_minus_xgb": -0.07363399232744983,
  "ci95_low": -0.09715625533538057,
  "ci95_high": -0.051734159832642165
 }
]

Positive = XGBoost has lower MAE. Intervals containing 0 mean the two models are not distinguishable on this data.


## Feature importance (split-based, training folds)

In [3]:
imp = pd.read_csv(RESULTS/'feature_importance.csv')
for m in ['Random Forest','XGBoost']:
    print(m); print(imp[imp.model==m].sort_values('mean_importance', ascending=False).round(3).to_string(index=False)); print()
d = imp[imp.feature.isin(['plan_area_density','plan_area_density_sq'])].groupby('model').mean_importance.sum()
print('Combined density + density^2 importance:'); print(d.round(3).to_string())

Random Forest
        model                  feature  mean_importance  std_importance
Random Forest        plan_area_density            0.391           0.058
Random Forest     plan_area_density_sq            0.376           0.040
Random Forest       wind_direction_deg            0.094           0.015
Random Forest            density_class            0.025           0.054
Random Forest horizontal_configuration            0.020           0.002
Random Forest             height_min_m            0.006           0.002
Random Forest           height_range_m            0.005           0.001
Random Forest             height_std_m            0.004           0.001
Random Forest             height_max_m            0.002           0.001
Random Forest    frac_taller_than_mean            0.002           0.000
Random Forest            configuration            0.001           0.001
Random Forest              height_skew            0.000           0.000

XGBoost
  model                  feature  mean_im

Density and density-squared are collinear, so importance is shared between them; together they hold roughly 77% (RF) to 82% (XGBoost) of split importance, with wind direction next. Importance describes tree usage, not physics.

## Prediction visualisation

![Observed and out-of-fold predictions](../reports/model_comparison.png)

![Feature importance](../reports/feature_importance.png)

## Model selection

There is **no winner** under family-held-out validation: XGBoost has the lower point estimate (MAE 0.224 vs 0.256 m/s) but the paired interval includes zero. Both beat Ridge and the mean baseline with non-overlapping point estimates. Trees predict constant values outside the training range, so they cannot extrapolate to the sparsest or densest family; see notebook 07.